In [ ]:
import csv
names = ['victim_name','victim_age','victim_gender','victim_race','victim_image_url','date','address','city','state','zip','county','agency_responsible','cause_of_death','description','disposition','charges','source','mentally_ill','armed']
f = open('data/data.csv', 'r')
reader = csv.DictReader(f, fieldnames=names)
data = [row for row in reader]
data.pop(0)


In [ ]:
import pandas

killings = pandas.DataFrame(data)
killings = killings.drop(None, 1)
killings.describe()

In [ ]:
killings['victim_gender'].value_counts()

In [ ]:
nonsense_genders = killings['victim_gender'].str.contains('White')
killings.loc[nonsense_genders, 'victim_gender'] = 'Unknown'
blank_genders = killings['victim_gender'].str.len() == 0
killings.loc[blank_genders, 'victim_gender'] = 'Unknown'
killings['victim_gender'].value_counts()

Not totally sure what to do about this duplicate problem yet.

In [ ]:
cities = pandas.read_csv('data/cities_corrected.csv')

# Normalizing, this is filtered on City, State so the state in the city name is extraneous
cities['City'] = cities['City'].str.replace('Kansas City Missouri', 'Kansas City')
cities

In [ ]:
columns = ['state_name', 'state', 'city', 'police_department', 'total', 'white', 'black', 'native', 'asian', 'hawaiian', 'asianpacificislander', 'other', 'two_races', 'hispanic', 'black_victims_2012', 
            'Unarmed Black',
       'Unarmed White', 'Unarmed Hispanic', 'Unarmed Native American',
       'Unarmed Asian/Pacific Islander', 'Unarmed Unknown', 'All Unarmed',
       'Rate of Unarmed Black', 'Rate of Unarmed White',
       'Rate of Unarmed Hispanic', 'Rate of Unarmed Native American',
       'Rate of Unarmed Asian/Pacific Islander', 'Rate of All Unarmed',
       'Black Victims', 'White Victims', 'Hispanic Victims',
       'Asian/Pacific Islander Victims', 'Victims of Unknown Race', 'All',
       'Percent of Victims Black', 'Rate of All Police Killings per 1M',
       'Rate of Police Killings of Black People per 1M',
       'Rate of Police Killings of White People per 1M',
       'Rate of Police Killings of Hispanic People per 1M',
       'Rate of Police Killings of Asian People per 1M', 'murder_rate',
       'violent_crime_rate', 'population', 'violent_crime',
       'murder_and_manslaughter', 'Black Armed', 'White Armed',
       'Hispanic Armed', 'Asian/Pacific Islander Armed', 'Unknown Armed',
       'Black Male Police Homicide Rate']
cities.columns = columns
relevant_columns = ['state_name', 'state', 'city', 'police_department', 'total', 'white', 'black', 'native', 'asian', 'hawaiian', 'asianpacificislander', 'other', 'two_races', 'hispanic', 'black_victims_2012',
                    'murder_rate','violent_crime_rate', 'population', 'violent_crime','murder_and_manslaughter']
selected_cities = cities[relevant_columns]
selected_cities

In [ ]:
# Normalize spelling
killings['agency_responsible'] = killings['agency_responsible'].str.replace('Polce', 'Police')
killings['agency_responsible'] = killings['agency_responsible'].str.replace('Charlotte-Mecklenberg', 'Charlotte-Mecklenburg')
killings['agency_responsible'] = killings['agency_responsible'].str.replace('Washington Metropolitan Police Department', 'DC Metropolitan Police Department')

# Multi- and mis-attributions
replacements = {
    'Brian Lipp': 'Toledo Police Department',
    'Kenneth Jewell Stafford': 'Reno Police Department',
    'Mark Jeffrey Kaplan': 'Phoenix Police Department',
    'Carl Lao': 'Stockton Police Department',
    'Jesse Gibbons': 'Lexington Police Department',
    'Patrick Heki': 'Las Vegas Metropolitan Police Department'
}

misattributions = killings[killings['victim_name'].isin(replacements.keys())]
# Make sure they all matched and the names are unique
assert misattributions['victim_name'].size == len(replacements.keys())
for name in replacements:
    killings.loc[killings['victim_name'] == name, 'agency_responsible'] = replacements[name]

# Shorten the names by using common abbreviations
killings['agency_responsible'] = killings['agency_responsible'].str.replace('Police Department', 'PD')
killings['agency_responsible'] = killings['agency_responsible'].str.replace('Metropolitan', 'Metro')
cities['police_department'] = cities['police_department'].str.replace('Police Department', 'PD')
cities['police_department'] = cities['police_department'].str.replace('Metropolitan', 'Metro')

# Normalize the text by removing ending periods and fancy apostrophes
killings['agency_responsible'] = killings['agency_responsible'].str.replace('?', '\'')
killings['agency_responsible'] = killings['agency_responsible'].str.replace('\.$', '')

city_departments = cities[['state', 'police_department']]
city_department_list = city_departments['police_department'].str.cat(city_departments['state'])

city_department_homicides = killings[killings['agency_responsible'].str.cat(killings['state']).isin(city_department_list)]
city_department_homicides.describe()

In [ ]:
city_department_homicides.to_csv(path_or_buf='./data/filtered.csv', index_label='key')
cities.to_csv(path_or_buf='./data/cities.csv', index_label='key')

In [ ]:
selected_cities['police_department'].value_counts()

In [ ]:
selected_cities

In [ ]:
def parseInt(x):
    try:
        return int(x.replace(',', ''))
    except AttributeError:
        return int(x)
selected_cities.columns

In [ ]:
computed_totals = selected_cities['total'].map(parseInt) / (selected_cities['white'].map(parseInt) + selected_cities['black'].map(parseInt) + selected_cities['native'].map(parseInt) + selected_cities['asian'].map(parseInt) + selected_cities['hawaiian'].map(parseInt) + selected_cities['other'].map(parseInt) + selected_cities['hispanic'].map(parseInt) + selected_cities['two_races'].map(parseInt))
# computed_totals = computed_totals.map(lambda x: abs(x - 1))
computed_totals.name = 'error'
selected_cities.join(computed_totals)[['city', 'error', 'hawaiian', 'asianpacificislander']].sort_values(by='error')

In [ ]:
from jupyter_core.paths import jupyter_data_dir
print(jupyter_data_dir())


In [ ]:
import sys
print(sys.executable)
